In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, roc_curve, confusion_matrix,
                              precision_recall_curve, ConfusionMatrixDisplay)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (7, 5)

RANDOM_STATE = 42

In [2]:
df = pd.read_csv("spambase_csv.csv")
df.rename(columns={df.columns[-1]: "class"}, inplace=True)
print("Shape of dataset:", df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'spambase_csv.csv'

In [ ]:
print("Total missing values in the dataset:", df.isnull().sum().sum())

if df.isnull().sum().sum() > 0:
    df = df.fillna(df.mean(numeric_only=True))
    print("Missing values were filled using column mean.")
else:
    print("No missing values found, dataset is clean.")

duplicates = df.duplicated().sum()
print("Duplicate rows found:", duplicates)

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x="class", data=df, palette=["#4C72B0", "#DD8452"])
plt.title("Class Distribution (0 = Not Spam, 1 = Spam)")
plt.xlabel("Class")
plt.ylabel("Number of Emails")
plt.show()

print(df["class"].value_counts(normalize=True) * 100)

In [ ]:
plt.figure(figsize=(10, 8))
corr = df.corr()
sns.heatmap(corr, cmap="coolwarm", center=0, xticklabels=False, yticklabels=False)
plt.title("Correlation Heatmap of All Features")
plt.show()

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif, chi2

X = df.drop("class", axis=1)
y = df["class"]

selector = SelectKBest(score_func=f_classif, k=6)
# selector = SelectKBest(score_func=chi2, k=6)
selector.fit(X, y)

important_features = X.columns[selector.get_support()].tolist()

print("Top Features:")
print(important_features)

df[important_features].hist(figsize=(14,8), bins=30, color="#4C72B0")
plt.suptitle("Histograms of Top Features")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
for i, feature in enumerate(important_features):
    sns.boxplot(x="class", y=feature, data=df, ax=axes[i], palette=["#4C72B0", "#DD8452"])
    axes[i].set_title(feature)
    axes[i].set_ylim(0, df[feature].quantile(0.95))
plt.show()

In [ ]:
X = df.drop(columns=["class"])
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

# Standard scaling
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_test_std = scaler.transform(X_test)

# Min-Max scaling 
mm_scaler = MinMaxScaler()
X_train_mm = mm_scaler.fit_transform(X_train)
X_test_mm = mm_scaler.transform(X_test)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)

In [ ]:
models = {}
train_times = {}

start = time.time()
gnb = GaussianNB()
gnb.fit(X_train_std, y_train)
train_times["GaussianNB"] = time.time() - start
models["GaussianNB"] = (gnb, X_test_std)

start = time.time()
mnb = MultinomialNB()
mnb.fit(X_train_mm, y_train)
train_times["MultinomialNB"] = time.time() - start
models["MultinomialNB"] = (mnb, X_test_mm)

start = time.time()
bnb = BernoulliNB()
bnb.fit(X_train_std, y_train)
train_times["BernoulliNB"] = time.time() - start
models["BernoulliNB"] = (bnb, X_test_std)

start = time.time()
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_std, y_train)
train_times["KNN"] = time.time() - start
models["KNN"] = (knn, X_test_std)

print("Models trained:", list(models.keys()))
print("Training times (seconds):", train_times)

In [ ]:
def evaluate_model(name, model, X_te, y_te):
    start = time.time()
    y_pred = model.predict(X_te)
    pred_time = time.time() - start

    y_proba = model.predict_proba(X_te)[:, 1]

    results = {
        "Model": name,
        "Accuracy": accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred),
        "Recall": recall_score(y_te, y_pred),
        "F1-score": f1_score(y_te, y_pred),
        "ROC-AUC": roc_auc_score(y_te, y_proba),
        "Prediction Time (s)": pred_time,
    }
    return results, y_pred, y_proba

results_list = []
predictions = {}  

for name, (model, X_te) in models.items():
    res, y_pred, y_proba = evaluate_model(name, model, X_te, y_test)
    results_list.append(res)
    predictions[name] = (y_pred, y_proba)

results_df = pd.DataFrame(results_list).set_index("Model")
results_df

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
axes = axes.flatten()

for ax, (name, (y_pred, y_proba)) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Not Spam", "Spam"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 6))
for name, (y_pred, y_proba) in predictions.items():
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_score = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc_score:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random Guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves - All Models")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(7, 6))
for name, (y_pred, y_proba) in predictions.items():
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    plt.plot(recall, precision, label=name)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves - All Models")
plt.legend()
plt.show()

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

k_values = list(range(1, 31, 2))

results = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_std, y_train)

    y_pred = knn.predict(X_test_std)

    results.append({
        "k": k,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-score": f1_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results)

results_df = results_df.round(4)

print(results_df)

best_row = results_df.loc[results_df["Accuracy"].idxmax()]
print(f"\nBest k = {best_row['k']}")
print(f"Best Accuracy = {best_row['Accuracy']:.4f}")

In [ ]:
tree_results = []

best_k=9

for algo in ["kd_tree", "ball_tree"]:
    knn_tree = KNeighborsClassifier(n_neighbors=best_k, algorithm=algo)

    start = time.time()
    knn_tree.fit(X_train_std, y_train)
    fit_time = time.time() - start

    start = time.time()
    y_pred_tree = knn_tree.predict(X_test_std)
    pred_time = time.time() - start

    tree_results.append({
        "Algorithm": algo,
        "Accuracy": accuracy_score(y_test, y_pred_tree),
        "Training Time (s)": fit_time,
        "Prediction Time (s)": pred_time,
    })

tree_df = pd.DataFrame(tree_results).set_index("Algorithm")
tree_df

In [ ]:
tree_df[["Training Time (s)", "Prediction Time (s)"]].plot(kind="bar", figsize=(7, 5), color=["#4C72B0", "#DD8452"])
plt.title("KDTree vs BallTree - Training & Prediction Time")
plt.ylabel("Time (seconds)")
plt.xticks(rotation=0)
plt.show()

In [ ]:
param_grid = {
    "n_neighbors": [3, 5, 7, 9, 11, 13, 15],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"],
}

grid_search = GridSearchCV(
    KNeighborsClassifier(),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)

start = time.time()
grid_search.fit(X_train_std, y_train)
grid_time = time.time() - start

print("Best parameters (GridSearchCV):", grid_search.best_params_)
print("Best cross-validated accuracy:", grid_search.best_score_)
print(f"GridSearchCV took {grid_time:.2f} seconds")

In [ ]:
cv_results = pd.DataFrame(grid_search.cv_results_)
pivot = cv_results.pivot_table(
    index="param_n_neighbors", columns="param_weights", values="mean_test_score", aggfunc="mean"
)

plt.figure(figsize=(6, 5))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlGnBu")
plt.title("GridSearchCV Accuracy\n(n_neighbors vs weights, averaged over metric)")
plt.ylabel("n_neighbors")
plt.xlabel("weights")
plt.show()

In [ ]:
from scipy.stats import randint

param_dist = {
    "n_neighbors": randint(1, 31),
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan", "chebyshev"],
}

random_search = RandomizedSearchCV(
    KNeighborsClassifier(),
    param_distributions=param_dist,
    n_iter=25,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

start = time.time()
random_search.fit(X_train_std, y_train)
random_time = time.time() - start

print("Best parameters (RandomizedSearchCV):", random_search.best_params_)
print("Best cross-validated accuracy:", random_search.best_score_)
print(f"RandomizedSearchCV took {random_time:.2f} seconds")

In [ ]:
random_cv_results = pd.DataFrame(random_search.cv_results_)

plt.figure(figsize=(8, 5))
sns.histplot(random_cv_results["mean_test_score"], bins=15, kde=True, color="#DD8452")
plt.xlabel("Mean CV Accuracy")
plt.ylabel("Number of Parameter Combinations Tried")
plt.title("RandomizedSearchCV - Distribution of CV Scores")
plt.show()

In [ ]:
search_comparison = pd.DataFrame({
    "Method": ["GridSearchCV", "RandomizedSearchCV"],
    "Best CV Accuracy": [grid_search.best_score_, random_search.best_score_],
    "Time Taken (s)": [grid_time, random_time],
    "Combinations Tried": [len(cv_results), len(random_cv_results)],
}).set_index("Method")

search_comparison

In [ ]:
best_knn = KNeighborsClassifier(**grid_search.best_params_)

start = time.time()
best_knn.fit(X_train_std, y_train)
train_times["KNN (Tuned)"] = time.time() - start

res, y_pred, y_proba = evaluate_model("KNN (Tuned)", best_knn, X_test_std, y_test)
results_list.append(res)
predictions["KNN (Tuned)"] = (y_pred, y_proba)

results_df = pd.DataFrame(results_list).set_index("Model")
results_df

In [ ]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_scores = {}
X_all_std = scaler.fit_transform(X)
X_all_mm = mm_scaler.fit_transform(X)

cv_scores["GaussianNB"] = cross_val_score(GaussianNB(), X_all_std, y, cv=cv_strategy, scoring="accuracy")
cv_scores["MultinomialNB"] = cross_val_score(MultinomialNB(), X_all_mm, y, cv=cv_strategy, scoring="accuracy")
cv_scores["BernoulliNB"] = cross_val_score(BernoulliNB(), X_all_std, y, cv=cv_strategy, scoring="accuracy")
cv_scores["KNN (Tuned)"] = cross_val_score(KNeighborsClassifier(**grid_search.best_params_), X_all_std, y, cv=cv_strategy, scoring="accuracy")

for name, scores in cv_scores.items():
    print(f"{name}: mean accuracy = {scores.mean():.4f}, std = {scores.std():.4f}")

In [ ]:
cv_df = pd.DataFrame(cv_scores)

plt.figure(figsize=(8, 5))
sns.boxplot(data=cv_df, palette="Set2")
sns.stripplot(data=cv_df, color="black", size=5, jitter=True)
plt.title("5-Fold Cross Validation Accuracy per Model")
plt.ylabel("Accuracy")
plt.show()

In [ ]:
train_time_df = pd.Series(train_times, name="Training Time (s)").sort_values()

plt.figure(figsize=(8, 5))
train_time_df.plot(kind="barh", color="#4C72B0")
plt.xlabel("Time (seconds)")
plt.title("Training Time Comparison")
plt.show()

In [ ]:
pred_time_df = results_df["Prediction Time (s)"].sort_values()

plt.figure(figsize=(8, 5))
pred_time_df.plot(kind="barh", color="#DD8452")
plt.xlabel("Time (seconds)")
plt.title("Prediction Time Comparison")
plt.show()

In [ ]:
plt.figure(figsize=(9, 6))
results_df["Accuracy"].sort_values().plot(kind="barh", color="#55A868")
plt.xlabel("Accuracy")
plt.title("Classifier Comparison - Test Accuracy")
plt.xlim(0.7, 1.0)
plt.show()

results_df.sort_values("Accuracy", ascending=False)